# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Group Name: UN_Group 5
#### Group members and ID:
* Nguyen Minh Duc (s4125574)
* Tran Si Anh Khoi (s4102087)
* Tran Le Huy (s4085023)

Environment: Python 3 and Jupyter notebook

Libraries used: 
* pandas
* numpy
* collections (Counter)
* sklearn
  * TfidfVectorizer
  * CountVectorizer
  * LogisticRegression
  * cross_val_score
* scipy
  * hstack
  * csr_matrix
* gensim
* warnings

## Introduction


## Importing libraries 

In [ ]:
# Code to import libraries as you need in this assessment, e.g.,
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from scipy.sparse import hstack, csr_matrix 
import gensim.downloader as api 
import warnings
warnings.filterwarnings('ignore')

## Task 2. Generating Feature Representations for Clothing Items Reviews

...... Sections and code blocks on buidling different document feature represetations


<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [ ]:
# Code to perform the task...


In [ ]:
df = pd.read_csv('processed.csv')
df.head()

In [ ]:
df.isna().sum()

In [ ]:
# Handle any NaN values (happens if a review was completely emptied by our filters in Task 1)
df['review_text'] = df['review_text'].fillna('')
df.isna().sum()

In [ ]:
# Load vocabulary into a dictionary for fast lookup
vocab_dict = {}
with open('vocab.txt', 'r', encoding='utf-8') as f:
    for line in f:
        word, idx = line.strip().split(':')
        vocab_dict[word] = int(idx)

In [ ]:
# 2. Generate Count Vectors (Bag-of-words)
with open('count_vectors.txt', 'w', encoding='utf-8') as f:
    for idx, row in df.iterrows():
        # Using review_id as the identifier requested by the assignment
        review_id = row['review_id'] 
        
        # Split the text string back into a list of words
        tokens = row['review_text'].split()
        
        # Count frequencies only for words that exist in our vocab
        token_counts = Counter([t for t in tokens if t in vocab_dict])
        
        if token_counts:
            # Sort by the integer index from the vocabulary for neatness
            sorted_counts = sorted([(vocab_dict[k], v) for k, v in token_counts.items()])
            # Format: word_integer_index:word_freq separated by comma
            vector_str = ",".join([f"{word_idx}:{freq}" for word_idx, freq in sorted_counts])
            f.write(f"#{review_id},{vector_str}\n")
        else:
            # Empty vector if no valid words
            f.write(f"#{review_id},\n")

In [ ]:
# 3. Load Word Embeddings & Calculate TF-IDF
# We use GloVe 100-dimensional vectors. It is reliable and relatively lightweight.
word_vectors = api.load("glove-wiki-gigaword-100")
# We restrict the TfidfVectorizer strictly to our Task 1 vocabulary
tfidf = TfidfVectorizer(vocabulary=vocab_dict)
tfidf_matrix = tfidf.fit_transform(df['review_text'])

In [ ]:
# 4. Generate Unweighted & Weighted Vectors
unweighted_file = open('unweighted_vectors.txt', 'w', encoding='utf-8')
weighted_file = open('weighted_vectors.txt', 'w', encoding='utf-8')

vector_size = word_vectors.vector_size

for idx, row in df.iterrows():
    review_id = row['review_id']
    tokens = row['review_text'].split()
    
    # Keep only tokens that are in our vocab AND exist in the GloVe model
    valid_tokens = [t for t in tokens if t in vocab_dict and t in word_vectors]
    
    if not valid_tokens:
        # If no valid tokens, output a vector of zeros
        zero_vec_str = ",".join(["0.0"] * vector_size)
        unweighted_file.write(f"#{review_id},{zero_vec_str}\n")
        weighted_file.write(f"#{review_id},{zero_vec_str}\n")
        continue

    # --- Unweighted Embeddings ---
    # Simple mathematical average of all word vectors in the review
    unweighted_vec = np.mean([word_vectors[t] for t in valid_tokens], axis=0)
    unweighted_file.write(f"#{review_id}," + ",".join(map(str, unweighted_vec)) + "\n")
    
    # --- Weighted Embeddings ---
    # Apply TF-IDF weights to the word embeddings
    doc_tfidf = tfidf_matrix[idx]
    weights = []
    vecs = []
    
    for t in valid_tokens:
        vocab_idx = vocab_dict[t]
        weight = doc_tfidf[0, vocab_idx] # Extract the TF-IDF score for this word
        weights.append(weight)
        vecs.append(word_vectors[t])
        
    if sum(weights) == 0:
        weighted_vec = unweighted_vec # Fallback safety measure
    else:
        # Calculate the weighted average using numpy
        weighted_vec = np.average(vecs, axis=0, weights=weights)
        
    weighted_file.write(f"#{review_id}," + ",".join(map(str, weighted_vec)) + "\n")

unweighted_file.close()
weighted_file.close()
print("Task 2 successfully completed! All 3 files are saved and ready.")

### Saving outputs
Save the count vector representation as per spectification.
- count_vectors.txt

In [ ]:
# code to save output data...

## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [ ]:
# Code to perform the task...


In [ ]:
df = pd.read_csv('processed.csv')

# Prepare Target Variable: Convert True/False in 'is_a_buyer' to 1/0
y = df['is_a_buyer'].astype(int)

In [ ]:
# Helper Function: Load Generated Vectors
def load_vector_file(filename, dim=100):
    vectors = []
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(',')
            # Skip the first element (the #review_id)
            vec = [float(x) for x in parts[1:] if x]
            
            # Safety check: if a row was empty, fill with zeros
            if len(vec) == 0:
                vec = [0.0] * dim
            vectors.append(vec)
    return np.array(vectors)

In [ ]:
# Q1: Language Model Comparisons
print("\n--- Running Q1: Language Model Comparisons ---")

# 1. Bag of Words (Using sklearn directly on our cleaned text for efficiency)
# We load the vocab to ensure it strictly matches Task 1
vocab_dict = {}
with open('vocab.txt', 'r', encoding='utf-8') as f:
    for line in f:
        word, idx = line.strip().split(':')
        vocab_dict[word] = int(idx)

df['review_text'] = df['review_text'].fillna('')
cv = CountVectorizer(vocabulary=vocab_dict)
X_bow = cv.fit_transform(df['review_text'])

In [ ]:
# 2. Unweighted Word Embeddings
X_unweighted = load_vector_file('unweighted_vectors.txt', dim=100)

In [ ]:
# 3. TF-IDF Weighted Word Embeddings
X_weighted = load_vector_file('weighted_vectors.txt', dim=100)

In [ ]:
# Initialize Classifier
clf = LogisticRegression(max_iter=1000)

# Perform 5-Fold Cross Validation
print("Evaluating Bag-of-Words...")
bow_scores = cross_val_score(clf, X_bow, y, cv=5, scoring='accuracy')

print("Evaluating Unweighted Vectors...")
unweighted_scores = cross_val_score(clf, X_unweighted, y, cv=5, scoring='accuracy')

print("Evaluating Weighted Vectors...")
weighted_scores = cross_val_score(clf, X_weighted, y, cv=5, scoring='accuracy')

print(f"-> Bag-of-Words Accuracy: {np.mean(bow_scores):.4f}")
print(f"-> Unweighted Word2Vec Accuracy: {np.mean(unweighted_scores):.4f}")
print(f"-> TF-IDF Weighted Word2Vec Accuracy: {np.mean(weighted_scores):.4f}")

In [ ]:
# Q2: Does more information provide higher accuracy?
print("\n--- Running Q2: Adding Extra Information ---")

# For this test, we will use the Bag-of-Words model as our baseline 
# (You can change this to X_weighted if it performed better in Q1)
X_baseline = X_bow
baseline_acc = np.mean(bow_scores)

In [ ]:
# --- Model 2: Description + Title ---
print("Evaluating Text + Title...")
df['review_title'] = df['review_title'].fillna('')
# Vectorize the title using TF-IDF (limit to top 1000 features to avoid noise)
title_tfidf = TfidfVectorizer(max_features=1000).fit_transform(df['review_title'])

# Combine Text features with Title features
X_text_title = hstack([X_baseline, title_tfidf])
text_title_scores = cross_val_score(clf, X_text_title, y, cv=5, scoring='accuracy')

In [ ]:
# --- Model 3: Description + Title + Extra Info ---
print("Evaluating Text + Title + Price & Rating...")
# Extract Price and Average Product Rating, filling any missing values with the median
price_arr = df['price'].fillna(df['price'].median()).values.reshape(-1, 1)
rating_arr = df['avg_product_rating'].fillna(df['avg_product_rating'].median()).values.reshape(-1, 1)

# Combine everything together into one large feature matrix
X_all_features = hstack([X_text_title, csr_matrix(price_arr), csr_matrix(rating_arr)])
all_features_scores = cross_val_score(clf, X_all_features, y, cv=5, scoring='accuracy')

In [ ]:
print(f"-> 1. Text Only Accuracy (Baseline): {baseline_acc:.4f}")
print(f"-> 2. Text + Title Accuracy: {np.mean(text_title_scores):.4f}")
print(f"-> 3. Text + Title + Extra Info (Price, Avg Rating) Accuracy: {np.mean(all_features_scores):.4f}")

## Summary
Give a short summary and anything you would like to talk about the assessment tasks here.

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>